In [ ]:
import tensorflow as tf
import tensorflow_text as text
import tensorflow_hub as hub

In [ ]:
bert_encoder_url="https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"
bert_preprocess_url="https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"

In [ ]:
#bert_preprocess_model=hub.KerasLayer(bert_preprocess_url)
bert_preprocess_model=hub.KerasLayer('bert_en_uncased_preprocess_3')
#bert_encoder_model=hub.KerasLayer(bert_encoder_url)
bert_encoder_model=hub.KerasLayer('bert_en_uncased_L-12_H-768_A-12_4')

In [ ]:
reviews =["One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.","i do not like the last part of the movie"]

In [ ]:
processed_text=bert_preprocess_model(reviews)

In [ ]:
processed_text.keys()

In [ ]:
processed_text['input_word_ids'][0]

In [ ]:
encoded_text=bert_encoder_model(processed_text)

In [ ]:
encoded_text.keys()

In [ ]:
encoded_text['pooled_output']

In [ ]:
encoded_text['sequence_output'][0][0]

In [ ]:
encoded_text['encoder_outputs'][-1]

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('dataset.csv')

In [ ]:
df.sample(10)


In [ ]:
df['output']=df['sentiment'].apply(lambda x : 1 if x=='positive' else 0)

In [ ]:
df.sample(10)

In [ ]:
df.groupby('output').describe()

In [ ]:
import seaborn as sns

In [ ]:
sns.countplot(x=df['sentiment'])

In [ ]:
positive_reviews=df[df['sentiment']=='positive']
negative_reviews=df[df['sentiment']=='negative']

In [ ]:
positive_reviews_100= positive_reviews.sample(100)
negative_reviews_100= positive_reviews.sample(100)

In [ ]:
df_200= pd.concat([positive_reviews_100, negative_reviews_100])

In [ ]:
df_200.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df_200['review'], 
    df_200['output'],
    stratify=df_200['output'],
    test_size=0.2
    )

In [ ]:
input_layer = tf.keras.layers.Input(shape=(), name="input", dtype=tf.dtypes.string)
preprocess_layer=bert_preprocess_model(input_layer)
encoder_layer = bert_encoder_model(preprocess_layer)
dropout_layer = tf.keras.layers.Dropout(rate=0.1, name ='dropout')(encoder_layer['pooled_output'])
output_layer = tf.keras.layers.Dense(1, activation='sigmoid', name = 'output')(dropout_layer)
model = tf.keras.Model(inputs = [input_layer], outputs=[output_layer])

In [ ]:
model.summary()

In [ ]:
metrics=[
    tf.keras.metrics.BinaryAccuracy(),
    tf.keras.metrics.Precision(),
    tf.keras.metrics.Recall()
    ]
model.compile(optimizer='adam', loss='binary_crossentropy', metrics= metrics)

In [ ]:
history=model.fit(x_train, y_train, validation_split=0.2, epochs=10, batch_size=10)

In [ ]:
y_predicted = model.predict(x_test)

In [ ]:
import numpy as np
y_predicted = np.where(y_predicted>0.5,1,0)

In [ ]:
model.evaluate(x_test,y_test)

In [ ]:
review=['Bad','i enjoy the movie']

In [ ]:
res=model.predict(review)
res

In [ ]:
res=np.where(res>0.5,1,0)

In [ ]:
res